In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer


# -------------------------------
# LOAD DATA
# -------------------------------
def load_data():
    xls = pd.ExcelFile("/kaggle/input/competitions/cohort-x-task-3/Task_3.xlsx")
    test = pd.read_excel(xls, sheet_name='Test')
    icd = pd.read_excel("/kaggle/input/competitions/cohort-x-task-3/mimic-iv_icd-10_dict.xlsx")
    return test, icd


# -------------------------------
# CLEAN TEXT
# -------------------------------
def clean_text(x):
    x = str(x).lower()
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def preprocess(test, icd):
    test["Condition_clean"] = test["Condition"].apply(clean_text)
    icd["long_title_clean"] = icd["long_title"].apply(clean_text)
    return test, icd


# -------------------------------
# EMBEDDINGS (MEDICAL MODEL)
# -------------------------------
def embed_texts(texts, model):
    return model.encode(
        texts,
        batch_size=128,
        convert_to_numpy=True,
        show_progress_bar=True
    )


# -------------------------------
# TF-IDF MATRIX
# -------------------------------
def compute_tfidf_matrix(icd, test):
    vectorizer = TfidfVectorizer(
        max_features=100000,
        ngram_range=(1, 2)
    )

    vectorizer.fit(list(icd["long_title_clean"]) + list(test["Condition_clean"]))
    icd_matrix = vectorizer.transform(icd["long_title_clean"])

    return vectorizer, icd_matrix


# -------------------------------
# FIXED LABEL ASSIGNMENT
# -------------------------------
def assign_labels_fixed(top_codes):
    keep = top_codes[:3]
    assoc = top_codes[3:8]
    diff = top_codes[8:15]

    fmt = lambda x: "; ".join(x) if x else "Not Applicable"

    return fmt(keep), fmt(assoc), fmt(diff)


# -------------------------------
# PREDICTION
# -------------------------------
def predict(test, icd, cond_emb, icd_emb, tfidf_vec, icd_matrix,
            top_k=20, alpha=0.9):

    results = []

    for idx, row in test.iterrows():

        # Embedding similarity
        cond_vector = cond_emb[idx].reshape(1, -1)
        emb_sims = cosine_similarity(cond_vector, icd_emb)[0]

        # TF-IDF similarity
        tfidf_vec_q = tfidf_vec.transform([row["Condition_clean"]])
        tfidf_sims = cosine_similarity(tfidf_vec_q, icd_matrix)[0]

        # Combine
        combined_sims = alpha * emb_sims + (1 - alpha) * tfidf_sims

        # Top candidates
        top_idx = combined_sims.argsort()[-top_k:][::-1]
        top_codes = icd.iloc[top_idx]["icd_code"].tolist()

        # Assign labels
        keep, assoc, diff = assign_labels_fixed(top_codes)

        results.append({
            "Condition": row["Condition"],
            "KEEP": keep,
            "ASSOCIATION": assoc,
            "DIFF": diff
        })

    return pd.DataFrame(results)


# -------------------------------
# MAIN
# -------------------------------
def main():
    test, icd = load_data()

    test, icd = preprocess(test, icd)

    # 🔥 Medical embedding model
    model = SentenceTransformer(
        'pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb'
    )

    cond_emb = embed_texts(test["Condition_clean"].tolist(), model)
    icd_emb = embed_texts(icd["long_title_clean"].tolist(), model)

    # Normalize embeddings (important)
    cond_emb = normalize(cond_emb)
    icd_emb = normalize(icd_emb)

    tfidf_vec, icd_matrix = compute_tfidf_matrix(icd, test)

    submission = predict(
        test,
        icd,
        cond_emb,
        icd_emb,
        tfidf_vec,
        icd_matrix,
        top_k=20,
        alpha=0.9
    )

    submission.to_csv("submission.csv", index=False)

    return submission


# -------------------------------
# RUN
# -------------------------------
submission = main()
print(submission.head())